# 🔊 Inferencia — HybridCNN Audio Classifier

Carga el modelo entrenado y predice sobre audios `.wav` / `.mp3` usando **exactamente el mismo pipeline** que `preprocess.py → main()` (sr=44100, n_mels=128, n_mfcc=13, n_fft=2048, hop=512, sin `target_duration`).

## 1 · Importaciones

In [1]:
from __future__ import annotations

import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import soundfile as sf

warnings.filterwarnings("ignore")

from src.utils.config import *
from src.models.hybrid_cnn_v2 import ImprovedMFCCCNN

print("✅ Importaciones completadas")

✅ Importaciones completadas


## 2 · Configuración

In [2]:
# ─────────────────────────────────────────────
# 🔧 RUTAS
# ─────────────────────────────────────────────
MODEL_PATH         = FINAL_MODEL_DIR / "best_mfcc_cnn.pt"
# MODEL_PATH       = CHECKPOINT_DIR / "checkpoint_epoch_1.pt"   # ← descomenta si usas checkpoint

LABEL_MAPPING_PATH = LABEL_MAPPING["human_label"]

AUDIO_FOLDER       = TEST_AUDIO_FOLDER   # carpeta con .wav / .mp3

# ─────────────────────────────────────────────
# 🎛️ PARÁMETROS DE AUDIO  (igual que preprocess.py → main())
# ─────────────────────────────────────────────
SAMPLE_RATE    = 44100
N_MELS         = 128
N_MFCC         = 13      # igual que preprocess
N_FFT          = 2048
HOP_LENGTH     = 512
PEAK_TARGET    = 0.99    # normalización de pico

# ─────────────────────────────────────────────
# 🧠 MODELO
# ─────────────────────────────────────────────
MODE           = "mel_mfcc"   # "mel_only" | "mfcc_only" | "mel_mfcc"
DROPOUT        = 0.25

# ─────────────────────────────────────────────
# 🖥️ INFERENCIA
# ─────────────────────────────────────────────
TOP_K          = 5       # cuántas predicciones mostrar por audio

print(f"📂 Carpeta de audios : {AUDIO_FOLDER}")
print(f"📦 Modelo            : {MODEL_PATH}")
print(f"🗂️  Label mapping      : {LABEL_MAPPING_PATH}")

📂 Carpeta de audios : /home/andres/Documentos/proyecto4geeks/tests/audios
📦 Modelo            : /home/andres/Documentos/proyecto4geeks/models/final/best_mfcc_cnn.pt
🗂️  Label mapping      : /home/andres/Documentos/proyecto4geeks/data/interim/processed_dataset/label_mapping_human_label.pkl


## 3 · Cargar label mapping

In [3]:
def load_label_mapping(path: Path) -> tuple[dict, dict, int]:
    """Devuelve (label2idx, idx2label, num_classes)."""
    with open(path, "rb") as f:
        payload = pickle.load(f)

    if isinstance(payload, dict) and "label2idx" in payload:
        label2idx = payload["label2idx"]
        idx2label = payload["idx2label"]
    else:
        label2idx = payload
        idx2label = {v: k for k, v in label2idx.items()}

    num_classes = len(label2idx)
    return label2idx, idx2label, num_classes


label2idx, idx2label, NUM_CLASSES = load_label_mapping(LABEL_MAPPING_PATH)

print(f"✅ {NUM_CLASSES} clases cargadas")
print("   Clases:", list(label2idx.keys()))

✅ 23 clases cargadas
   Clases: ['alert_sirem', 'ambient_noise', 'another_animal', 'breathing', 'car_crash', 'construction_noise', 'crime', 'crying', 'dog', 'domestic_activity', 'doors', 'engine', 'explosion', 'fight', 'fire', 'glass_metal', 'gun_shot', 'instrument', 'machine', 'traffic', 'voice', 'water', 'weather']


## 4 · Construir transforms (idénticos a `preprocess.py → main()`)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️  Dispositivo: {device}")

mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS,
    power=2.0,
).to(device)

amplitude_to_db = T.AmplitudeToDB(stype="power").to(device)

mfcc_transform = T.MFCC(
    sample_rate=SAMPLE_RATE,
    n_mfcc=N_MFCC,
    melkwargs={
        "n_fft": N_FFT,
        "hop_length": HOP_LENGTH,
        "n_mels": N_MELS,
        "center": True,
        "power": 2.0,
    },
).to(device)

# Cache de resamplers para no crear uno por audio
_resampler_cache: dict[int, T.Resample] = {}

def get_resampler(orig_sr: int) -> T.Resample | None:
    if orig_sr == SAMPLE_RATE:
        return None
    if orig_sr not in _resampler_cache:
        _resampler_cache[orig_sr] = T.Resample(orig_freq=orig_sr, new_freq=SAMPLE_RATE).to(device)
    return _resampler_cache[orig_sr]

print("✅ Transforms construidos")

🖥️  Dispositivo: cuda
✅ Transforms construidos


## 5 · Pipeline de preprocesado (idéntico a `preprocess.py`)

In [5]:
def load_and_preprocess(audio_path: Path) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Replica exactamente Preprocess.process_single_audio():
      1. Carga con soundfile
      2. Mono
      3. Resamplea a SAMPLE_RATE si hace falta
      4. Sin recorte de duración (target_duration=None)
      5. Normalización de pico
      6. MelSpec + AmplitudeToDB
      7. MFCC

    Returns
    -------
    mel  : [1, N_MELS, T]   float32
    mfcc : [1, N_MFCC, T]   float32
    """
    # ── Carga ────────────────────────────────────────────────────────────
    audio_np, sr = sf.read(str(audio_path))

    waveform = torch.tensor(audio_np, dtype=torch.float32)

    if waveform.ndim == 1:
        waveform = waveform.unsqueeze(0)          # [1, T]
    else:
        waveform = waveform.transpose(0, 1)       # [channels, T]

    # ── Mono ─────────────────────────────────────────────────────────────
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    waveform = waveform.to(device)

    # ── Resampleo ─────────────────────────────────────────────────────────
    resampler = get_resampler(sr)
    if resampler is not None:
        waveform = resampler(waveform)

    # ── Normalización de pico ─────────────────────────────────────────────
    peak = waveform.abs().max().clamp_min(1e-8)
    waveform = waveform / peak * PEAK_TARGET

    # ── Features ──────────────────────────────────────────────────────────
    with torch.inference_mode():
        mel    = amplitude_to_db(mel_transform(waveform))  # [1, N_MELS, T]
        mfcc   = mfcc_transform(waveform)                  # [1, N_MFCC, T]

    return mel.float(), mfcc.float()


print("✅ Función de preprocesado lista")

✅ Función de preprocesado lista


## 6 · Cargar modelo

In [6]:
def detect_mode_from_state_dict(state_dict: dict) -> str:
    keys = set(state_dict.keys())
    has_mel  = any(k.startswith("cnn_mel.")  for k in keys)
    has_mfcc = any(k.startswith("cnn_mfcc.") for k in keys)

    if has_mel and has_mfcc:
        return "mel_mfcc"
    elif has_mel:
        return "mel_only"
    elif has_mfcc:
        return "mfcc_only"
    else:
        raise ValueError("No se encontraron keys cnn_mel.* ni cnn_mfcc.* en el state_dict.")


def load_model(model_path: Path, num_classes: int, dropout: float) -> tuple[ImprovedMFCCCNN, str]:
    checkpoint = torch.load(model_path, map_location=device)

    if isinstance(checkpoint, dict) and "model_state" in checkpoint:
        state_dict = checkpoint["model_state"]
        epoch_info = checkpoint.get("epoch", "?")
        acc_info   = checkpoint.get("best_acc", "?")
        print(f"   📌 Checkpoint — epoch: {epoch_info}  |  best_acc: {acc_info}")
    else:
        state_dict = checkpoint

    detected_mode = detect_mode_from_state_dict(state_dict)
    print(f"   🔍 Modo detectado automáticamente: {detected_mode}")

    model = ImprovedMFCCCNN(num_classes=num_classes, dropout=dropout, mode=detected_mode).to(device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, detected_mode


model, MODE = load_model(MODEL_PATH, NUM_CLASSES, DROPOUT)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Modelo cargado  |  {total_params:,} parámetros  |  modo: {MODE}")

   🔍 Modo detectado automáticamente: mel_only
✅ Modelo cargado  |  1,180,375 parámetros  |  modo: mel_only


## 7 · Función de predicción

In [7]:
@torch.inference_mode()
def predict(audio_path: Path, top_k: int = TOP_K) -> dict:
    """
    Retorna un dict con:
      - filename    : nombre del archivo
      - prediction  : clase predicha (str)
      - confidence  : probabilidad de la clase predicha (float)
      - top_k       : lista de (clase, prob) para las top_k predicciones
      - logits_raw  : tensor de logits (para debugging)
    """
    mel, mfcc = load_and_preprocess(audio_path)  # [1, F, T]

    # Añadir dimensión de batch → [1, 1, F, T]
    mel_b  = mel.unsqueeze(0)
    mfcc_b = mfcc.unsqueeze(0)

    if MODE == "mel_only":
        logits = model(mel_b)
    elif MODE == "mfcc_only":
        logits = model(mel_b, mfcc_b)   # mel ignorado internamente
    else:  # mel_mfcc
        logits = model(mel_b, mfcc_b)

    probs = torch.softmax(logits, dim=-1).squeeze(0).cpu()

    top_probs, top_idxs = probs.topk(min(top_k, len(idx2label)))

    pred_idx   = int(top_idxs[0])
    pred_label = idx2label[pred_idx]
    pred_conf  = float(top_probs[0])

    top_list = [(idx2label[int(i)], float(p)) for i, p in zip(top_idxs, top_probs)]

    return {
        "filename"   : audio_path.name,
        "prediction" : pred_label,
        "confidence" : pred_conf,
        "top_k"      : top_list,
        "logits_raw" : logits.squeeze(0).cpu(),
    }


print("✅ Función predict() lista")

✅ Función predict() lista


## 8 · Probar un audio individual (opcional)

In [8]:
# ── Cambia esto al archivo que quieras probar ───────────────────────────
# SINGLE_AUDIO = Path("/ruta/al/audio.wav")
# ────────────────────────────────────────────────────────────────────────

# Demo: coge el primer audio de la carpeta si existe
audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if audios:
    SINGLE_AUDIO = audios[0]
    result = predict(SINGLE_AUDIO)

    print(f"\n🔊 Archivo    : {result['filename']}")
    print(f"🏆 Predicción : {result['prediction']}")
    print(f"📊 Confianza  : {result['confidence']:.2%}")
    print(f"\n📋 Top-{TOP_K}:")
    for rank, (label, prob) in enumerate(result["top_k"], 1):
        bar = "█" * int(prob * 30)
        print(f"  {rank}. {label:<25} {prob:.2%}  {bar}")
else:
    print(f"⚠️  No hay audios .wav/.mp3 en {AUDIO_FOLDER}")


🔊 Archivo    : 11325622-police-siren-sound-effect-240674.mp3
🏆 Predicción : alert_sirem
📊 Confianza  : 91.37%

📋 Top-5:
  1. alert_sirem               91.37%  ███████████████████████████
  2. traffic                   2.62%  
  3. weather                   1.30%  
  4. instrument                0.85%  
  5. fight                     0.66%  


## 9 · Inferencia por lotes sobre toda la carpeta

In [9]:
from tqdm import tqdm  # en vez de from tqdm.notebook import tqdm

audios = sorted(AUDIO_FOLDER.glob("**/*"))
audios = [a for a in audios if a.suffix.lower() in (".wav", ".mp3")]

if not audios:
    raise FileNotFoundError(f"No se encontraron audios en {AUDIO_FOLDER}")

print(f"📂 {len(audios)} audios encontrados en {AUDIO_FOLDER}\n")

rows = []
errors = []

for audio_path in tqdm(audios, desc="Procesando audios"):
    try:
        result = predict(audio_path)
        row = {
            "filename"   : result["filename"],
            "prediction" : result["prediction"],
            "confidence" : result["confidence"],
        }
        # Añadir columna por cada clase del top-k
        for label, prob in result["top_k"]:
            row[f"prob_{label}"] = round(prob, 4)
        rows.append(row)
    except Exception as e:
        errors.append({"filename": audio_path.name, "error": str(e)})
        print(f"❌ Error en {audio_path.name}: {e}")

results_df = pd.DataFrame(rows)

print(f"\n✅ Procesados: {len(rows)}  |  Errores: {len(errors)}")
results_df.head(10)

📂 6 audios encontrados en /home/andres/Documentos/proyecto4geeks/tests/audios



Procesando audios: 100%|██████████| 6/6 [00:00<00:00, 54.57it/s]


✅ Procesados: 6  |  Errores: 0


,filename,prediction,confidence,prob_alert_sirem,prob_traffic,prob_weather,prob_instrument,prob_fight,prob_explosion,prob_gun_shot,prob_construction_noise,prob_domestic_activity,prob_car_crash,prob_dog,prob_breathing,prob_voice,prob_another_animal,prob_engine,prob_fire,prob_glass_metal
0,11325622-police-siren-sound-effect-240674.mp3,alert_sirem,0.913749,0.9137,0.0262,0.0130,0.0085,0.0066,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,audio_607a0.mp3,explosion,0.515416,NaN,NaN,NaN,NaN,NaN,0.5154,0.3114,0.0453,0.0176,0.0157,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,dragon-studio-dog-barking-406629.mp3,dog,0.986189,NaN,NaN,0.0008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.9862,0.0031,0.0016,0.0011,NaN,NaN,NaN
3,freesound_community-dog-barking-70772.mp3,dog,0.989566,NaN,NaN,0.0012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.9896,0.0009,0.0015,0.0013,NaN,NaN,NaN
4,freesound_community-m4-assault-rifle-long-burs...,another_animal,0.181064,NaN,NaN,NaN,NaN,NaN,0.1382,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.1811,0.1772,0.1339,0.1232
5,m4a1_unsil-1_d6515.mp3,explosion,0.842647,NaN,NaN,0.0030,NaN,NaN,0.8426,0.1286,NaN,NaN,NaN,NaN,NaN,NaN,0.0055,NaN,0.0026,NaN


## 10 · Resumen de predicciones

In [10]:
if not results_df.empty:
    summary = (
        results_df
        .groupby("prediction")
        .agg(
            count=("filename", "count"),
            avg_confidence=("confidence", "mean"),
        )
        .sort_values("count", ascending=False)
        .reset_index()
    )
    summary["avg_confidence"] = summary["avg_confidence"].map("{:.2%}".format)
    print("📊 Distribución de predicciones:")
    display(summary)

    # Archivos con confianza baja (puede necesitar revisión)
    CONFIDENCE_THRESHOLD = 0.50
    low_conf = results_df[results_df["confidence"] < CONFIDENCE_THRESHOLD]
    if not low_conf.empty:
        print(f"\n⚠️  {len(low_conf)} audios con confianza < {CONFIDENCE_THRESHOLD:.0%}:")
        display(low_conf[["filename", "prediction", "confidence"]])

📊 Distribución de predicciones:


,prediction,count,avg_confidence
0,explosion,2,67.90%
1,dog,2,98.79%
2,another_animal,1,18.11%
3,alert_sirem,1,91.37%



⚠️  1 audios con confianza < 50%:


,filename,prediction,confidence
4,freesound_community-m4-assault-rifle-long-burs...,another_animal,0.181064


## 11 · Exportar resultados a CSV (opcional)

In [11]:
OUTPUT_CSV = AUDIO_FOLDER / "predictions.csv"

if not results_df.empty:
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Resultados guardados en: {OUTPUT_CSV}")
else:
    print("⚠️  No hay resultados para exportar.")

✅ Resultados guardados en: /home/andres/Documentos/proyecto4geeks/tests/audios/predictions.csv
